# RPC Proiect - 1.1. Topspin

Donea Fernando-Emanuel

grupa 243

## ex 1







In [1]:

import math

class Nod:
    def __init__(self, info, parinte=None, g=0, h=0, detalii_mutare=None):
        self.info = info
        self.parinte = parinte
        self.g = g
        self.h = h
        self.f = g + h
        self.detalii_mutare = detalii_mutare

    def obtine_drum(self):
        drum = []
        nod = self
        while nod is not None:
            drum.append(nod)
            nod = nod.parinte
        drum.reverse()
        return drum

    def in_drum(self, info):
        nod = self
        while nod is not None:
            if nod.info == info:
                return True
            nod = nod.parinte
        return False

    def __lt__(self, other):

        #in caz de egalitate, descrescator dupa g
        if self.f==other.f:
            return self.g>other.g
        # ordonam crescator dupa f
        return self.f<other.f

    def __repr__(self):
        return f"Nod(info={self.info}, g={self.g}, h={self.h}, f={self.f})"


class Graf:
    def __init__(self,n,k,start=None,):
        self.n=n
        self.k=k
        self.SCOP_TOPSPIN=[i for i in range(1,n+1)]


        banda=start.info
        while banda[0]!=1:
            #rotiam la dreapta banda circulara pana cand avem 1 pe prima poz
            aux=banda[1:]+[banda[0]]
            banda=aux

        if self.valideaza(banda) is not True:
            print("Lista de start este invalida")
        else:
            h_start=self.estimeaza_h(banda)
            self.start=Nod(info=banda,h=h_start)


    def valideaza(self, banda):
        if len(banda) is not self.n:
            return False

        if sorted(set(banda)) != self.SCOP_TOPSPIN:
            return False

        return True


    def scop_func(self, nod):
        if nod.info == self.SCOP_TOPSPIN:
            return True
        else:
            return False


    def succesori(self, nod):
        lista_succesori = []
        banda_curenta=nod.info

        for i in range(self.n):
            banda=list(banda_curenta)

            #inversam cele k piese de pe turnichet
            for j in range(self.k//2):
                st=(i+j)%self.n
                dr=(i+self.k-j-1)%self.n

                # interschimbam capetele
                aux=banda[st]
                banda[st]=banda[dr]
                banda[dr]=aux

            #rotim banda pana avem 1 pe prima pozitie
            while banda[0]!=1:
                banda=banda[1:]+[banda[0]]

            if banda!=banda_curenta:
                g_suc=nod.g+1
                h_suc=self.estimeaza_h(banda)
                succesor=Nod(info=banda,parinte=nod,g=g_suc,h=h_suc)

                lista_succesori.append(succesor)

        return lista_succesori



    def estimeaza_h(self, banda):
        greseli=0

        for i in range(self.n):
            piesa_curenta=banda[i]
            piesa_urmatoare=banda[(i+1)%self.n]

            #o pereche e buna doar daca nr sunt consecutive sau daca se inchide cercu
            diferenta=abs(piesa_curenta-piesa_urmatoare)
            if not (diferenta==1 or diferenta==self.n-1):
                greseli+=1

        return math.ceil(greseli/2)

    def afisareDrum(self, drum):
        if drum_A is not None:
            print(f"Nr de pasi: {len(drum)-1}")
            for i, nod in enumerate(drum):
                if(i==0):
                    print(f"Start: {nod}")
                else:
                    print({nod})
        else:
            print("Nu exista")





**Justificare euristica**

Spunem ca o euristica este admisibila daca $\hat{h}(nod) \leq h(nod)$

Euristica propusa calculeaza numarul de greseli adiacente gresite neorientate de pe banda si il imparte la 2:

$\hat{h}(nod)=ceil(greseli/2)$

La fiecare mutare, intoarcerea turnichetului modifica vecinii doar de la captul secventei, rupand si refacand exact 2 legaturi cu restul bandei. Deoarce consideram adiacentele indiferent de sens, legaturile din interiorul turnichetului isi schimba doar ordinea, ramanand valide si neschimband numarul de greseli. Prin urmare o singura mutare de cost=1 poate repara cel mult 2 greseli.

Orice pereche de nr adiacente `(banda[i], banda[i+1])` care nu se afla in ordinea secventiala a benzii este considerata o greseala. Cu alte cuvinte, doua nr sunt asezate corect pe banda daca diferenta lor in modul este egala cu 1 sau cu n-1 (pentru a inchide circular banda). Orice alta pereche este considerata o greseala.

Avand in vedere ca la fiecare pas putem corecta maxim 2 greseli, vom avea nevoie de cel putin $ceil(greseli/2)$ mutari pentru ajunge la starea scop, unde greseli=0. Astfel, formula returneaza mereu o valoarea mai mica sau egala decat costul real, eurisitca fiind admisibila.


In [2]:
stare_initiala=[7,3,9,2,6,1,5,4,8]
nod=Nod(info=stare_initiala)

Topspin=Graf(n=9,k=3,start=nod)



# ex 2 - A*

In [3]:
import heapq

def A_star(graf):
    frontiera=[] #openlist
    heapq.heappush(frontiera, graf.start)

    #initilizam multimea de stari explorate cu multimea vida
    vizitate=set() #closedlist


    while len(frontiera)>0:
        nod_curent= heapq.heappop(frontiera)

        #daca este scop returnam solutia
        if graf.scop_func(nod_curent)==True:
            return nod_curent.obtine_drum()

        #daca nodul e vizitat continuam
        stare_curenta=tuple(nod_curent.info) #transformam nodul in tuplu ca sa o puteam aduga in set
        if stare_curenta in vizitate:
            continue

        #marcam nodul curent ca vizitat
        vizitate.add(stare_curenta)

        #expandam nodul
        succesori=graf.succesori(nod_curent)
        for s in succesori:
            stare_succesor=tuple(s.info)

            #verifcam sa nu l fi vizitt deja
            if stare_succesor not in vizitate:
                heapq.heappush(frontiera,s)

    return None


In [4]:
drum_A=A_star(Topspin)
Topspin.afisareDrum(drum_A)



Nr de pasi: 9
Start: Nod(info=[1, 5, 4, 8, 7, 3, 9, 2, 6], g=0, h=4, f=4)
{Nod(info=[1, 6, 2, 5, 4, 8, 7, 3, 9], g=1, h=3, f=4)}
{Nod(info=[1, 6, 2, 5, 7, 8, 4, 3, 9], g=2, h=3, f=5)}
{Nod(info=[1, 6, 2, 5, 7, 8, 9, 3, 4], g=3, h=3, f=6)}
{Nod(info=[1, 6, 7, 5, 2, 8, 9, 3, 4], g=4, h=3, f=7)}
{Nod(info=[1, 6, 7, 5, 9, 8, 2, 3, 4], g=5, h=3, f=8)}
{Nod(info=[1, 6, 7, 8, 9, 5, 2, 3, 4], g=6, h=2, f=8)}
{Nod(info=[1, 6, 7, 8, 9, 3, 2, 5, 4], g=7, h=2, f=9)}
{Nod(info=[1, 4, 5, 6, 7, 8, 9, 3, 2], g=8, h=1, f=9)}
{Nod(info=[1, 2, 3, 4, 5, 6, 7, 8, 9], g=9, h=0, f=9)}


# ex 3 IDA

In [5]:
inf=10000000000000000

def IDA(graf):

    limita=graf.start.f
    nod_start=graf.start


    # DFS recursiv pe ramura limitata la costul f
    def expandeaza(nod_curent, limita):

        #daca nodul curent are f>limita atunci nu expandam si returnam noul threshold
        if nod_curent.f>limita:
            return nod_curent.f

        #daca este scop returnam solutia
        if graf.scop_func(nod_curent)==True:
            return nod_curent.obtine_drum()

        minim_limita=inf
        succesori=graf.succesori(nod_curent)
        for s in succesori:
            #filtram succesorii pentru a evira reintoarcea in drum
            if not nod_curent.in_drum(s.info):
                rezultat=expandeaza(s,limita)

                #daca rezultatul e lista am gsait o solutie
                if isinstance(rezultat,list):
                    return rezultat

                #dca nu am gasit solutia pe ramura asta, actualizam limita
                if rezultat<minim_limita:
                    minim_limita=rezultat

        return minim_limita

    while True:
        rezultat=expandeaza(nod_start,limita)

        if rezultat==inf:
            return None

        if isinstance(rezultat, list):
            return rezultat

        limita=rezultat



In [6]:
drum_IDA=IDA(Topspin)
Topspin.afisareDrum(drum_IDA)


Nr de pasi: 9
Start: Nod(info=[1, 5, 4, 8, 7, 3, 9, 2, 6], g=0, h=4, f=4)
{Nod(info=[1, 8, 4, 5, 7, 3, 9, 2, 6], g=1, h=4, f=5)}
{Nod(info=[1, 8, 7, 5, 4, 3, 9, 2, 6], g=2, h=3, f=5)}
{Nod(info=[1, 8, 7, 3, 4, 5, 9, 2, 6], g=3, h=3, f=6)}
{Nod(info=[1, 8, 7, 3, 9, 5, 4, 2, 6], g=4, h=4, f=8)}
{Nod(info=[1, 6, 2, 8, 7, 3, 9, 5, 4], g=5, h=4, f=9)}
{Nod(info=[1, 6, 7, 8, 2, 3, 9, 5, 4], g=6, h=3, f=9)}
{Nod(info=[1, 6, 7, 8, 9, 3, 2, 5, 4], g=7, h=2, f=9)}
{Nod(info=[1, 4, 5, 6, 7, 8, 9, 3, 2], g=8, h=1, f=9)}
{Nod(info=[1, 2, 3, 4, 5, 6, 7, 8, 9], g=9, h=0, f=9)}
